### PCA-Nets and Allen-Cahn Problem
Next, we consider the following differential equation:
\begin{align*}
    \epsilon \Delta u + (u^3-u)&= f &&\text{ in } (0,1)^2, \\
    n \cdot \nabla u &= 0 &&\text{ on } \partial (0,1)^2.
\end{align*}
We want to learn the mapping $f \mapsto u$. We use a data driven approach, where we have a dataset of pairs $(f_i, u_i)$, and we want to learn a neural network that approximates the solution operator.

We try this with the PCA-Net idea, where we first compute the PCA of the input and output data, and then learn a mapping between the PCA coefficients.

In [ ]:
# This block is for GPU selection. Please execute.
# import pathlib
# import os
# user = int(str(pathlib.Path().resolve())[22:24])
# os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

save_path = "/localdata/tomfre/AllenCahn_data"

First we load the input and output data for the Allen-Cahn problem.

In [ ]:
import qewton

f_data = qewton.DEFAULT_DL_BACKEND.load_data(f"{save_path}/input_data.pt")
u_data = qewton.DEFAULT_DL_BACKEND.load_data(f"{save_path}/output_data.pt")

Next up are the variables again:

In [ ]:
F = qewton.Variable("f", dim=1) 
U = qewton.Variable("u", dim=1)

In [ ]:
# Split data into training and testing:
total_data = len(u_data)
train_N = int(0.8*total_data)

# TODO: Build the training and test set, as in the last example
# We want to split the data into training and testing sets, at the number train_N, use
# the [:] syntax
train_f = f_data[:train_N]
train_u = u_data[:train_N]

test_f = f_data[train_N:]
test_u = u_data[train_N:]

# TODO: Create the test dataset:
dataset = qewton.data.GridDataSet(
    data=[train_f, train_u], 
    feature_variables=[F, U],
)
test_dataset = qewton.data.

In [ ]:
# A data loader like before
data_loader = qewton.data.DataLoader(
    data_set=dataset,
    test_data_set=test_dataset,
    batch_size=4000,
    splitting_ratio=(1.0, 0.0, 0.0),
    shuffle_data=False,
)

We can visualize the training data using the data loader.

In [ ]:
qewton.viz.Figure(data_loader.visualize(mode=qewton.optim.EvaluationPhase.TRAIN)).show()

Now we build the PCANet model for our problem.

We have to decide on how many principal components to use in the PCANet model.
Select for the input 25 and for the output 400.

In [ ]:
# TODO: Build the PCANet model for the Allen-Cahn problem.
# Add the input and output variable, and set the input pca and output pca dimensions to 25 and 400
# respectively.
model = qewton.algorithms.PCANet(
    input_variable=,
    output_variable=,
    pca_n_input=,
    pca_n_output=,
    data_source_node=data_loader,
    fcn_hidden_layers=2,
    fcn_hidden_neurons=50,
)

Now we have to build the trainings pipeline:

In [ ]:
# TODO: Define the MSE constraint
constraint = qewton.constraints.

computation_pipeline = qewton.Graph()
# The pipeline is really simple:
with computation_pipeline.tracker(0):
    f, u = data_loader()
    model_output = # Evaluate the model on the input f
    constraint # plug the data and model output into the constraint

computation_pipeline.setup()

Lets look at the pipeline:

In [ ]:
# TODO: Select different depths and show what nodes appear
qewton.viz.Figure(computation_pipeline.diagram(depth=0, show_edge_labels=False)).show()

Now training as always:

In [9]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.01,
    max_iterations=5000,
    lr_scheduler=qewton.optim.StepLR(step_size=1000, gamma=0.25)
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase],
    graphs=[computation_pipeline],
    training_objectives=[constraint],
    device=qewton.cuda(0),
)

trainer.run()

Optimization Phase 1: 100%|██████████| 5000/5000 [05:18<00:00, 15.71it/s, loss=0.0462]


In [10]:
layout = computation_pipeline.visualize(
    model.output_ports[0],
    reference=data_loader.get_output_port(U),
    mode=qewton.optim.EvaluationPhase.TEST,
)

fig = qewton.viz.Figure(layout)
app = qewton.viz.DashApplication.create(fig)
app.run(port=8070, jupyter_mode="external")

Dash app running on http://127.0.0.1:8070/
